# 05 - Reference CNN Check

Bounded CPU smoke checks and a short feasibility measurement only. This is not an experiment or full training run: no learning curves, accuracy claims or selected model checkpoint are produced. Later experiments must initialise a fresh model after reseeding, never reuse these updated weights.

## 1. Fixed inputs, CPU and real loaders

Use the saved split and encoding unchanged, plus the existing training augmentation and deterministic validation preprocessing. Seed Python, NumPy and torch with 42. Keep batch size 32, zero workers, all tail batches and existing CPU thread settings.

In [1]:
from pathlib import Path
from datetime import datetime
from importlib.metadata import version
import hashlib
import json
import platform
import sys
import tempfile
import time
import statistics
import torch
from torch import nn
from torch.utils.data import DataLoader

def find_project_root(start):
    start=Path(start).resolve()
    for candidate in (start,*start.parents):
        if (candidate/"requirements.txt").is_file() and (candidate/"notebooks/00_Environment_Check.ipynb").is_file():
            return candidate
    raise FileNotFoundError("Open the project root or notebooks directory.")

PROJECT_ROOT=find_project_root(Path.cwd())
assert find_project_root(PROJECT_ROOT/"notebooks")==PROJECT_ROOT
assert Path(sys.prefix).resolve()==PROJECT_ROOT/".venv"
assert Path(sys.executable).resolve().parent==PROJECT_ROOT/".venv/Scripts"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0,str(PROJECT_ROOT))
from src.data_pipeline import FacialFeatureDataset, seed_everything
from src.models import ReferenceCNN

SEED=42
DEVICE=torch.device("cpu")
seed_everything(SEED)
checks={}
def require(name,condition):
    checks[name]=bool(condition)
    assert checks[name],name
    print(name+": PASS",flush=True)

def digest(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

protected=["data/processed/split_manifest.csv","data/processed/class_mapping.json",
           "requirements.txt","requirements-lock.txt","src/data_pipeline.py"]
before_hashes={name:digest(PROJECT_ROOT/name) for name in protected}
mapping=json.loads((PROJECT_ROOT/protected[1]).read_text(encoding="utf-8"))
loading_evidence=json.loads((PROJECT_ROOT/"outputs/metrics/data_loading_check.json").read_text(encoding="utf-8"))
require("saved_loading_checks_passed",not loading_evidence["failed_checks"])
train_dataset=FacialFeatureDataset(PROJECT_ROOT,"train")
validation_dataset=FacialFeatureDataset(PROJECT_ROOT,"validation")
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True,
                        generator=torch.Generator().manual_seed(SEED),num_workers=0,drop_last=False)
validation_loader=DataLoader(validation_dataset,batch_size=32,shuffle=False,num_workers=0,drop_last=False)
require("saved_dataset_and_loader_counts",len(train_dataset)==50927 and len(validation_dataset)==12733 and len(train_loader)==1592 and len(validation_loader)==398)
initial_threads=torch.get_num_threads()
initial_interop_threads=torch.get_num_interop_threads()
print("Device:",DEVICE,"CPU threads:",initial_threads,"Interop threads:",initial_interop_threads)


saved_loading_checks_passed: PASS


saved_dataset_and_loader_counts: PASS


Device: cpu CPU threads: 8 Interop threads: 8


## 2. Architecture, shapes and one optimiser-step smoke check

Each padded 3x3 convolution learns local features; ReLU adds nonlinearity and 2x2 max pooling halves spatial dimensions. Global average pooling produces 128 features, dropout regularises training only, and the linear head produces eight raw logits. Cross-entropy consumes those logits directly, with no preceding softmax. Hooks record shapes without retaining extra activations.

In [2]:
model=ReferenceCNN().to(DEVICE)
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)
architecture=str(model)
total_parameters=sum(parameter.numel() for parameter in model.parameters())
trainable_parameters=sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
print(model)
print("Total parameters:",total_parameters,"Trainable:",trainable_parameters)
require("specified_parameter_count",total_parameters==trainable_parameters==94280)
require("model_is_cpu",all(parameter.device.type=="cpu" for parameter in model.parameters()))
images,labels,image_ids=next(iter(train_loader))
fixed_images=images.to(DEVICE)
labels=labels.to(DEVICE)
require("input_tensor_contract",list(fixed_images.shape)==[32,3,224,224] and fixed_images.dtype==torch.float32 and bool(torch.isfinite(fixed_images).all()) and float(fixed_images.min())>=0 and float(fixed_images.max())<=1)
require("label_contract",labels.dtype==torch.int64 and bool(((labels>=0)&(labels<8)).all()))
shape_trace=[{"layer":"input","shape":list(fixed_images.shape)}]
handles=[]
def record_shape(name):
    def hook(module,inputs,output):
        shape_trace.append({"layer":name,"type":type(module).__name__,"shape":list(output.shape)})
    return hook
for name,layer in model.named_modules():
    if name and not list(layer.children()):
        handles.append(layer.register_forward_hook(record_shape(name)))
model.train()
optimizer.zero_grad(set_to_none=True)
try:
    logits=model(fixed_images)
finally:
    for handle in handles:
        handle.remove()
print(json.dumps(shape_trace,indent=2))
require("forward_logits_shape_and_finite",list(logits.shape)==[32,8] and bool(torch.isfinite(logits).all()))
loss=criterion(logits,labels)
smoke_loss=float(loss.detach())
require("cross_entropy_finite",bool(torch.isfinite(loss)))
loss.backward()
gradients=[parameter.grad for parameter in model.parameters() if parameter.grad is not None]
require("gradients_finite",bool(gradients) and all(bool(torch.isfinite(gradient).all()) for gradient in gradients))
require("some_gradient_nonzero",any(bool(torch.any(gradient!=0)) for gradient in gradients))
before_step={name:parameter.detach().clone() for name,parameter in model.named_parameters()}
optimizer.step()
require("adam_changes_parameters",any(not torch.equal(before_step[name],parameter.detach()) for name,parameter in model.named_parameters()))
del logits,loss,gradients,before_step
optimizer.zero_grad(set_to_none=True)
print("One-batch smoke cross-entropy:",smoke_loss,"(not a performance result)")


ReferenceCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): AdaptiveAvgPool2d(output_size=1)
    (1): Flatten(start_dim=1, end_dim=-1)
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=8, bias=True)
  )
)
Total parameters: 94280 Trainable: 94280
specified_parameter_count: PASS


model_is_cpu: PASS


input_tensor_contract: PASS


label_contract: PASS


[
  {
    "layer": "input",
    "shape": [
      32,
      3,
      224,
      224
    ]
  },
  {
    "layer": "features.0",
    "type": "Conv2d",
    "shape": [
      32,
      32,
      224,
      224
    ]
  },
  {
    "layer": "features.1",
    "type": "ReLU",
    "shape": [
      32,
      32,
      224,
      224
    ]
  },
  {
    "layer": "features.2",
    "type": "MaxPool2d",
    "shape": [
      32,
      32,
      112,
      112
    ]
  },
  {
    "layer": "features.3",
    "type": "Conv2d",
    "shape": [
      32,
      64,
      112,
      112
    ]
  },
  {
    "layer": "features.4",
    "type": "ReLU",
    "shape": [
      32,
      64,
      112,
      112
    ]
  },
  {
    "layer": "features.5",
    "type": "MaxPool2d",
    "shape": [
      32,
      64,
      56,
      56
    ]
  },
  {
    "layer": "features.6",
    "type": "Conv2d",
    "shape": [
      32,
      128,
      56,
      56
    ]
  },
  {
    "layer": "features.7",
    "type": "ReLU",
    "shape": [
 

cross_entropy_finite: PASS


gradients_finite: PASS


some_gradient_nonzero: PASS


adam_changes_parameters: PASS


One-batch smoke cross-entropy: 2.093776226043701 (not a performance result)


## 3. Evaluation and temporary checkpoint round trip

Use `model.eval()` and `torch.inference_mode()` together. Evaluate the same fixed tensor twice, then save only a temporary state dictionary, reload it into the same architecture and compare logits. Delete only the uniquely created temporary checkpoint in `finally`; no model artifact is retained.

In [3]:
model.eval()
with torch.inference_mode():
    evaluation_logits=model(fixed_images)
    repeated_logits=model(fixed_images)
require("eval_mode_and_inference_tensors",not model.training and not evaluation_logits.requires_grad and evaluation_logits.is_inference())
require("repeated_fixed_evaluation_identical",torch.equal(evaluation_logits,repeated_logits))
checkpoint_path=None
try:
    with tempfile.NamedTemporaryFile(dir=PROJECT_ROOT/"outputs/models",prefix="reference_cnn_smoke_",suffix=".pt",delete=False) as temporary:
        checkpoint_path=Path(temporary.name)
    torch.save(model.state_dict(),checkpoint_path)
    reloaded=ReferenceCNN().to(DEVICE)
    reloaded.load_state_dict(torch.load(checkpoint_path,map_location=DEVICE,weights_only=True))
    reloaded.eval()
    with torch.inference_mode():
        reloaded_logits=reloaded(fixed_images)
    require("state_dict_round_trip_logits_identical",torch.equal(evaluation_logits,reloaded_logits))
finally:
    if checkpoint_path is not None:
        checkpoint_path.unlink(missing_ok=True)
require("only_temporary_checkpoint_cleaned",checkpoint_path is not None and not checkpoint_path.exists())
del reloaded,reloaded_logits,evaluation_logits,repeated_logits


eval_mode_and_inference_tensors: PASS


repeated_fixed_evaluation_identical: PASS


state_dict_round_trip_logits_identical: PASS


only_temporary_checkpoint_cleaned: PASS


## 4. Bounded CPU feasibility timing

The approximately 120-second budget includes two requested warm-up training batches, iterator creation and loading. Then measure up to five training and three validation batches. Every individual time starts before `next(iterator)`, so loading is included. Training performs forward, cross-entropy, backward and Adam; validation uses evaluation/inference mode. Finite checks are included in the measurement.

Check elapsed time between batches. Reserve 30 seconds for validation and use the last batch duration as a conservative admission estimate. A single unexpectedly slow batch cannot be interrupted safely, so a small overrun is possible. Record actual completions and stop reasons; never run a full epoch. No thread settings are changed.

In [4]:
TIME_BUDGET=120.0
VALIDATION_RESERVE=30.0
warmup_times=[]
training_times=[]
validation_times=[]
timing_stop_reasons=[]
started=time.perf_counter()
training_iterator=iter(train_loader)

def training_step(iterator):
    # Loading is deliberately inside the caller's timed interval.
    batch_images,batch_labels,_=next(iterator)
    optimizer.zero_grad(set_to_none=True)
    output=model(batch_images.to(DEVICE))
    batch_loss=criterion(output,batch_labels.to(DEVICE))
    if not bool(torch.isfinite(output).all()) or not bool(torch.isfinite(batch_loss)):
        raise ValueError("Non-finite training logits/loss in bounded timing")
    batch_loss.backward()
    optimizer.step()

model.train()
for batch in range(2):
    elapsed=time.perf_counter()-started
    expected=warmup_times[-1] if warmup_times else 0.0
    if elapsed+expected>=TIME_BUDGET:
        timing_stop_reasons.append("Warm-up stopped at the wall-time budget.")
        break
    batch_start=time.perf_counter()
    training_step(training_iterator)
    warmup_times.append(time.perf_counter()-batch_start)
    print(f"Warm-up {len(warmup_times)}: {warmup_times[-1]:.3f}s",flush=True)

if len(warmup_times)==2:
    for batch in range(5):
        elapsed=time.perf_counter()-started
        expected=(training_times or warmup_times)[-1]
        if elapsed+expected>=TIME_BUDGET-VALIDATION_RESERVE:
            timing_stop_reasons.append("Timed training stopped to retain the budget and validation reserve.")
            break
        batch_start=time.perf_counter()
        training_step(training_iterator)
        training_times.append(time.perf_counter()-batch_start)
        print(f"Timed training {len(training_times)}: {training_times[-1]:.3f}s",flush=True)
else:
    timing_stop_reasons.append("Timed training omitted because two warm-ups did not fit.")

model.eval()
if time.perf_counter()-started<TIME_BUDGET:
    validation_iterator=iter(validation_loader)
    with torch.inference_mode():
        for batch in range(3):
            elapsed=time.perf_counter()-started
            expected=validation_times[-1] if validation_times else 0.0
            if elapsed+expected>=TIME_BUDGET:
                timing_stop_reasons.append("Validation stopped at the wall-time budget.")
                break
            batch_start=time.perf_counter()
            batch_images,batch_labels,_=next(validation_iterator)
            output=model(batch_images.to(DEVICE))
            validation_loss=criterion(output,batch_labels.to(DEVICE))
            if not bool(torch.isfinite(output).all()) or not bool(torch.isfinite(validation_loss)):
                raise ValueError("Non-finite validation logits/loss in bounded timing")
            validation_times.append(time.perf_counter()-batch_start)
            print(f"Timed validation {len(validation_times)}: {validation_times[-1]:.3f}s",flush=True)
else:
    timing_stop_reasons.append("No budget remained for validation.")
timing_elapsed=time.perf_counter()-started
train_mean=statistics.mean(training_times) if training_times else None
validation_mean=statistics.mean(validation_times) if validation_times else None
rough_epoch_seconds=(train_mean*len(train_loader)+validation_mean*len(validation_loader)) if train_mean is not None and validation_mean is not None else None
require("bounded_batch_counts",len(warmup_times)<=2 and len(training_times)<=5 and len(validation_times)<=3)
require("timing_did_not_run_a_full_epoch",len(training_times)+len(warmup_times)<len(train_loader) and len(validation_times)<len(validation_loader))
require("validation_timing_used_eval_inference",not model.training and (not validation_times or (output.is_inference() and not output.requires_grad)))
print("Timing elapsed:",round(timing_elapsed,3),"seconds")
print("Mean timed training batch:",train_mean,"Mean validation batch:",validation_mean)
print("Rough projected train+validation epoch (seconds):",rough_epoch_seconds)
print("Stop reasons:",timing_stop_reasons)


Warm-up 1: 1.432s


Warm-up 2: 1.117s


Timed training 1: 1.194s


Timed training 2: 1.143s


Timed training 3: 1.055s


...

Timed validation 3: 0.479s


bounded_batch_counts: PASS


timing_did_not_run_a_full_epoch: PASS


validation_timing_used_eval_inference: PASS


Timing elapsed: 9.484 seconds
Mean timed training batch: 1.1032950599998004 Mean validation batch: 0.46710183333683136
Rough projected train+validation epoch (seconds): 1942.352265187741
Stop reasons: []


## 5. Save evidence and discard smoke-test weights

The epoch estimate is a rough extrapolation from a short sample using actual loader batch counts (including partial tails). Loading, verification overhead, cache state, CPU contention and sustained performance may differ during a full epoch. It is not a measured full-epoch time or evidence of model quality.

Delete the live updated model and optimiser after this check. A later experiment must call `seed_everything(42)` and instantiate a new `ReferenceCNN()` and optimiser; no smoke-test weights are saved as a starting point.

In [5]:
after_hashes={name:digest(PROJECT_ROOT/name) for name in protected}
require("split_mapping_dependencies_pipeline_unchanged",before_hashes==after_hashes)
require("thread_settings_unchanged",torch.get_num_threads()==initial_threads and torch.get_num_interop_threads()==initial_interop_threads)
limitations=[
    "Rough full-epoch projection from a short CPU sample, not a measured epoch or a performance/accuracy claim.",
    "Batch loading and finite checks are included; sustained CPU speed, contention and filesystem caching can change timings.",
    "The budget is checked between batches; one unexpectedly slow batch can overrun it.",
    "Last partial batches are projected at full-batch mean cost; warm-up is excluded from the projection.",
    "No reliable animal/source groups exist; unidentified related crops may cross the fixed partitions.",
    "Smoke-test parameter updates are discarded; initialise a fresh seeded model and optimiser for later experiments.",
]
summary={
    "checked_at":datetime.now().astimezone().isoformat(),
    "architecture":architecture,"intermediate_shapes":shape_trace,
    "parameters":{"total":total_parameters,"trainable":trainable_parameters},
    "seed":SEED,"device":str(DEVICE),"cpu_threads":initial_threads,"cpu_interop_threads":initial_interop_threads,
    "optimizer":{"name":"Adam","learning_rate":0.001},"loss":"CrossEntropyLoss on raw logits; no softmax",
    "smoke_cross_entropy":smoke_loss,
    "runtime":{"python":platform.python_version(),"torch":torch.__version__,"torchvision":version("torchvision"),
               "numpy":version("numpy"),"pandas":version("pandas"),"Pillow":version("Pillow"),"ipykernel":version("ipykernel"),
               "platform":platform.platform(),"processor":platform.processor()},
    "loaders":{"batch_size":32,"num_workers":0,"drop_last":False,"train_shuffle":True,"validation_shuffle":False,
               "train_images":len(train_dataset),"validation_images":len(validation_dataset),
               "train_batches":len(train_loader),"validation_batches":len(validation_loader)},
    "timing":{
        "budget_seconds":TIME_BUDGET,"validation_reserve_seconds":VALIDATION_RESERVE,
        "includes_loading":True,"budget_includes_warmup":True,
        "warmup_training_batch_seconds":warmup_times,"timed_training_batch_seconds":training_times,"timed_validation_batch_seconds":validation_times,
        "completed":{"warmup_training":len(warmup_times),"timed_training":len(training_times),"timed_validation":len(validation_times)},
        "training_mean_seconds":train_mean,"validation_mean_seconds":validation_mean,
        "phase_elapsed_seconds":timing_elapsed,"budget_overrun_seconds":max(0.0,timing_elapsed-TIME_BUDGET),
        "stop_reasons":timing_stop_reasons,
    },
    "rough_epoch_projection":{
        "label":"Rough projection, not a measured full epoch",
        "formula":"mean_training_batch_seconds * actual_train_loader_batches + mean_validation_batch_seconds * actual_validation_loader_batches",
        "seconds":rough_epoch_seconds,"minutes":rough_epoch_seconds/60 if rough_epoch_seconds is not None else None,
    },
    "temporary_checkpoint_removed":not checkpoint_path.exists(),
    "optimizer_step_counts":{"smoke":1,"warmup":len(warmup_times),"timed":len(training_times)},
    "protected_file_hashes_before":before_hashes,"protected_file_hashes_after":after_hashes,
    "checks":checks,"failed_checks":[name for name,passed in checks.items() if not passed],"limitations":limitations,
    "execution":{"python_executable":sys.executable,"kernel_name":"enge707-assessment3","working_directory":str(Path.cwd())},
}
# Remove updated model references rather than leaving a reusable starting point.
optimizer.zero_grad(set_to_none=True)
del model,optimizer,layer
globals().pop("output",None)
summary["smoke_model_and_optimizer_discarded"]=True
EVIDENCE_PATH=PROJECT_ROOT/"outputs/metrics/reference_cnn_check.json"
EVIDENCE_PATH.write_text(json.dumps(summary,indent=2)+"\n",encoding="utf-8")
assert json.loads(EVIDENCE_PATH.read_text(encoding="utf-8"))["checks"]==checks
print("Saved:",EVIDENCE_PATH.relative_to(PROJECT_ROOT))
print("Smoke-test weights discarded. Initialise a fresh seeded model for future experiments.")


split_mapping_dependencies_pipeline_unchanged: PASS


thread_settings_unchanged: PASS


Saved: outputs\metrics\reference_cnn_check.json
Smoke-test weights discarded. Initialise a fresh seeded model for future experiments.
